# Bill Swift — Lead Scoring & Revenue Intelligence

An end-to-end automation project for **Bill Swift**, a simulated usage-based billing SaaS company modeled on Chargebee.

**Pipeline**: Webinar form → this scoring/prediction logic → Make.com webhook → HubSpot / Salesforce / email automation.

In [1]:
import os
import re
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timezone
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", None)

## 1. Configuration & Scoring Constants

The 8-signal model (100 points total) and every category below are grounded in Chargebee's real published data, not guessed:

| Signal | Points | Grounding |
|---|---|---|
| Email Domain Authority | 8 | Personal email domains score 0 |
| Monthly Billing Volume | 20 | $100K/mo is Chargebee's real Performance→Enterprise crossover |
| Primary Revenue Challenge | 20 | Tied to the real "Mastering Usage-Based Billing" webinar content |
| Current Billing Tool | 12 | Highest score for highest-pain "switching from" situations |
| Job Title | 12 | Chargebee is explicitly finance-led (CFO/RevOps = top tier) |
| Company Size | 12 | Mirrors Chargebee's real Starter→Enterprise growth arc |
| CRM Tech Stack | 8 | Salesforce & HubSpot are Chargebee's real native integrations |
| Industry | 8 | Chargebee's real customer base is >51% software/IT + AI + e-learning |

In [2]:
MAKE_WEBHOOK_URL = os.environ.get(
    "MAKE_WEBHOOK_URL",
    "https://hook.us2.make.com/jnnny7ok4185bsx1of1tb2z3jvsylobt"
)

TRAINING_FILE_PATH = os.environ.get(
    "TRAINING_FILE_PATH",
    "/Users/praveen/Desktop/BillSwift/billswift_testing_dataset_final.xlsx"
)

FREE_EMAIL_DOMAINS = {
    "gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "aol.com",
    "icloud.com", "live.com", "mail.com", "protonmail.com", "zoho.com"
}

REQUIRED_FIELDS = [
    "first_name", "last_name", "email",
    "company_name", "current_billing_tool", "monthly_billing_volume",
    "job_title", "crm_stack", "company_size", "industry",
    "primary_revenue_challenge"
]

JOB_TITLE_SCORES = {
    "cfo_head_of_finance": 12,
    "cro_head_of_revops": 12,
    "ceo_founder": 8,
    "director": 8,
    "manager": 4,
    "other": 0,
}

CRM_STACK_SCORES = {
    "salesforce": 8,
    "hubspot_enterprise": 8,
    "hubspot_other": 4,
    "other_none": 0,
}

COMPANY_SIZE_SCORES = {
    "1000_plus": 12,
    "200_999": 8,
    "20_199": 4,
    "under_20": 0,
}

INDUSTRY_SCORES = {
    "b2b-saas": 8,
    "ai-cloud": 8,
    "edtech": 8,
    "ecommerce": 4,
    "fintech": 4,
    "proptech-iot": 4,
    "digital-media": 0,
    "other": 0,
}

PRIMARY_CHALLENGE_SCORES = {
    "manual-no-automation": 20,
    "revenue-recognition-compliance": 20,
    "migrating-flat-to-usage": 12,
    "hybrid-tiered-complexity": 12,
    "metering-tracking-accuracy": 6,
    "other": 0,
}

_model = None
_model_columns = None

## 2. Data Cleaning & Helper Functions

In [3]:
def clean_data(data):
    """Sanitizes raw form payload string entries."""
    c = {k: v.strip() if isinstance(v, str) else v for k, v in data.items()}
    if c.get("first_name"):   c["first_name"]   = c["first_name"].title()
    if c.get("last_name"):    c["last_name"]    = c["last_name"].title()
    if c.get("company_name"): c["company_name"] = c["company_name"].title()
    if c.get("email"):        c["email"]        = c["email"].lower()
    return c

def is_valid_email_syntax(email):
    """RFC 5322-style regex validation."""
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", email))

def get_email_domain(email):
    if "@" not in email:
        return ""
    return email.split("@")[-1].strip().lower()

def bucket_transaction_volume(raw_volume):
    """
    Converts a raw numeric monthly_billing_volume into the same bucket
    strings used in the training dataset (under-50k / 50k-250k / 250k-1m /
    1m-10m / 10m+). Required so the ML model's dummy-encoded categorical
    feature actually matches at prediction time -- without this, a raw
    number like 75000 would create a brand-new, unseen dummy column and
    get silently zeroed out, meaning volume would contribute nothing to
    the revenue prediction.
    """
    try:
        v = float(raw_volume)
    except (ValueError, TypeError):
        v = 0
    if v < 50000:
        return "under-50k"
    elif v < 250000:
        return "50k-250k"
    elif v < 1000000:
        return "250k-1m"
    elif v < 10000000:
        return "1m-10m"
    else:
        return "10m+"

## 3. 8-Signal Scoring Engine (100 pts)

Cold 0–35 · Warm 36–65 · Hot 66–100, with a separate MQL qualification gate at score ≥ 85.

In [4]:
def calculate_lead_score(payload):
    score = 0
    email = payload.get("email", "").lower()
    domain = get_email_domain(email)
    is_personal_email = domain in FREE_EMAIL_DOMAINS

    # Signal 1: Email Domain Authority (8 pts)
    if not is_personal_email and domain != "":
        score += 8

    # Signal 2: Monthly Billing Volume (20 pts)
    volume = payload.get("monthly_billing_volume", 0)
    try:
        volume = float(volume)
    except (ValueError, TypeError):
        volume = 0
    if volume > 100000:
        score += 20
    elif volume >= 50000:
        score += 12
    else:
        score += 4

    # Signal 3: Current Billing Tool (12 pts)
    tool = str(payload.get("current_billing_tool", "")).lower()
    if tool in ["legacy-enterprise", "in-house", "none-manual"]:
        # Highest pain / highest switching intent -- best-fit prospects
        score += 12
    elif tool in ["stripe-paypal"]:
        # Has payments but no real billing layer -- moderate opportunity
        score += 6
    elif tool in ["modern-platform"]:
        # Likely already on a competitor's modern billing platform -- hardest to win
        score += 2
    else:
        score += 0

    # Signal 4: Job Title (12 pts) -- dropdown value, direct lookup
    title = str(payload.get("job_title", "other")).lower()
    score += JOB_TITLE_SCORES.get(title, 0)

    # Signal 5: CRM Tech Stack (8 pts) -- dropdown value, direct lookup
    crm = str(payload.get("crm_stack", "other_none")).lower()
    score += CRM_STACK_SCORES.get(crm, 0)

    # Signal 6: Company Size (12 pts) -- dropdown value, direct lookup
    size = str(payload.get("company_size", "under_20")).lower()
    score += COMPANY_SIZE_SCORES.get(size, 0)

    # Signal 7: Industry (8 pts) -- dropdown value, direct lookup
    industry = str(payload.get("industry", "other")).lower()
    score += INDUSTRY_SCORES.get(industry, 0)

    # Signal 8: Primary Revenue Challenge (20 pts)
    challenge = str(payload.get("primary_revenue_challenge", "other")).lower()
    score += PRIMARY_CHALLENGE_SCORES.get(challenge, 0)

    score = min(score, 100)
    tier = "Hot" if score >= 66 else ("Warm" if score >= 36 else "Cold")
    qualification_status = "MQL" if score >= 85 else "Nurture"

    return score, tier, qualification_status, is_personal_email

### Quick sanity check
Best-case lead should hit exactly 100; a weak lead should land in Cold with `is_personal_email=True`.

In [5]:
best_case = {
    "email": "cfo@bigcorp.com", "monthly_billing_volume": 150000,
    "current_billing_tool": "legacy-enterprise", "job_title": "cfo_head_of_finance",
    "crm_stack": "salesforce", "company_size": "1000_plus", "industry": "ai-cloud",
    "primary_revenue_challenge": "manual-no-automation"
}
weak_case = {
    "email": "someone@gmail.com", "monthly_billing_volume": 2000,
    "current_billing_tool": "modern-platform", "job_title": "other",
    "crm_stack": "other_none", "company_size": "under_20", "industry": "other",
    "primary_revenue_challenge": "other"
}
print("Best case:", calculate_lead_score(best_case))
print("Weak case:", calculate_lead_score(weak_case))

Best case: (100, 'Hot', 'MQL', False)
Weak case: (6, 'Cold', 'Nurture', True)


## 4. Random Forest Revenue Predictor

Trained on `billswift_testing_dataset.xlsx` — a synthetic dataset generated to match this exact schema, with `Annual Revenue` grounded in Chargebee's real plan economics 

In [6]:
def train_model():
    global _model, _model_columns
    if not os.path.exists(TRAINING_FILE_PATH):
        print("WARNING: Training file not found. Revenue prediction disabled.")
        return None, None
    try:
        df = pd.read_excel(TRAINING_FILE_PATH, engine="openpyxl")
        FEAT = ["Company Size", "Monthly Transaction Volume Bucket", "Current Billing Solution", "Lead Score"]
        for col in FEAT[:-1]:
            if col in df.columns:
                df[col] = df[col].fillna("missing")
        X = pd.get_dummies(df[FEAT], columns=FEAT[:-1])
        y = df["Annual Revenue"]
        m = RandomForestRegressor(n_estimators=100, random_state=42)
        m.fit(X, y)
        print(f"Model trained successfully on {len(df)} rows.")
        return m, list(X.columns)
    except Exception as e:
        print(f"Error training ML model: {e}")
        return None, None

def predict_revenue(data, score):
    global _model, _model_columns
    if _model is None:
        return None
    try:
        row = {
            "Company Size": data.get("company_size", "missing"),
            "Monthly Transaction Volume Bucket": bucket_transaction_volume(data.get("monthly_billing_volume", 0)),
            "Current Billing Solution": data.get("current_billing_tool", "missing"),
            "Lead Score": score
        }
        df_row = pd.DataFrame([row])
        enc = pd.get_dummies(df_row, columns=["Company Size", "Monthly Transaction Volume Bucket", "Current Billing Solution"])
        aligned = enc.reindex(columns=_model_columns, fill_value=0)
        return round(float(_model.predict(aligned)[0]), 2)
    except Exception as e:
        print(f"Prediction failed: {e}")
        return None

_model, _model_columns = train_model()

Model trained successfully on 250 rows.


In [7]:
print("Best-case predicted revenue:", predict_revenue(best_case, calculate_lead_score(best_case)[0]))
print("Weak-case predicted revenue: ", predict_revenue(weak_case, calculate_lead_score(weak_case)[0]))

Best-case predicted revenue: 69101.45
Weak-case predicted revenue:  10947.55


## 5. Explore the Training Dataset

A quick look at what the synthetic dataset actually contains -- useful for portfolio screenshots and sanity-checking the score/revenue relationship.

In [8]:
df = pd.read_excel(TRAINING_FILE_PATH)
df.head(10)

,Timestamp,Name,Email,Company Name,Company Domain,Industry,Industry Other,Job Title,Job Title Other,Primary Revenue Challenge,Primary Revenue Challenge Other,CRM Stack,CRM Stack Other,Company Size,Monthly Transaction Volume,Current Billing Solution,Heard From,Lead Score,Lead Tier,Qualification Status,Is Personal Email,Annual Revenue,Monthly Transaction Volume Bucket
0,2026-08-08 04:16:58 UTC,Nicholas Miller,rebecca.green@walker-wagner.com,Walker-Wagner,walker-wagner.com,ecommerce,NaN,manager,NaN,other,Customer self-serve billing portal,hubspot_enterprise,NaN,under_20,2600,in-house,linkedin,40,Warm,Nurture,False,11281.22,under-50k
1,2026-08-08 04:21:37 UTC,Sandra Ibarra,nancy.scott@pattersongarnerandfowler.com,"Patterson, Garner and Fowler",pattersongarnerandfowler.com,b2b-saas,NaN,director,NaN,manual-no-automation,NaN,hubspot_other,NaN,1000_plus,1015000,none-manual,newsletter,92,Hot,MQL,False,59678.19,1m-10m
2,2026-08-08 04:21:48 UTC,Jennifer Reed,zachary.stephens@wrightandsons.com,Wright and Sons,wrightandsons.com,other,Construction Tech,cfo_head_of_finance,NaN,revenue-recognition-compliance,NaN,other_none,Freshsales,200_999,7700,modern-platform,linkedin,54,Warm,Nurture,False,25239.76,under-50k
3,2026-08-08 04:16:34 UTC,Marco Morris,kevin.bailey@foxandsons.com,Fox and Sons,foxandsons.com,ai-cloud,NaN,cro_head_of_revops,NaN,migrating-flat-to-usage,NaN,hubspot_enterprise,NaN,200_999,551400,none-manual,partner,88,Hot,MQL,False,72615.30,250k-1m
4,2026-08-08 04:22:09 UTC,Gavin Freeman,jay.powell@allen-jones.com,Allen-Jones,allen-jones.com,ai-cloud,NaN,other,Product Manager,revenue-recognition-compliance,NaN,salesforce,NaN,under_20,1500,legacy-enterprise,linkedin,60,Warm,Nurture,False,16360.41,under-50k
5,2026-08-08 04:21:36 UTC,Mark Price,tiffany.castillo@douglas-green.com,Douglas-Green,douglas-green.com,ai-cloud,NaN,other,Product Manager,hybrid-tiered-complexity,NaN,hubspot_other,NaN,under_20,158500,stripe-paypal,google,58,Warm,Nurture,False,28346.31,50k-250k
6,2026-08-08 04:21:35 UTC,Jordan Price,joshua.mcguire@kinggroup.com,King Group,kinggroup.com,other,Gaming,cfo_head_of_finance,NaN,revenue-recognition-compliance,NaN,hubspot_other,NaN,20_199,38600,none-manual,linkedin,64,Warm,Nurture,False,20841.95,under-50k
7,2026-08-08 04:18:54 UTC,Charles Williams,austin.russell@ballardinc.com,Ballard Inc,ballardinc.com,ecommerce,NaN,director,NaN,revenue-recognition-compliance,NaN,hubspot_enterprise,NaN,200_999,900,none-manual,linkedin,72,Hot,Nurture,False,32423.38,under-50k
8,2026-08-08 04:20:49 UTC,Jessica Cooper,anthony.butler@jones-williams.com,Jones-Williams,jones-williams.com,ai-cloud,NaN,other,Engineering Lead,manual-no-automation,NaN,salesforce,NaN,1000_plus,156300,legacy-enterprise,newsletter,88,Hot,MQL,False,54337.56,50k-250k
9,2026-08-08 04:16:11 UTC,Antonio Porter,benjamin.brown@mejia-mcdonald.com,Mejia-Mcdonald,mejia-mcdonald.com,b2b-saas,NaN,other,Sales Rep,migrating-flat-to-usage,NaN,hubspot_enterprise,NaN,under_20,15200,stripe-paypal,partner,46,Warm,Nurture,False,12150.69,under-50k


In [9]:
print("Lead tier distribution:")
print(df["Lead Tier"].value_counts())
print()
print("Score vs. Annual Revenue correlation:", round(df["Lead Score"].corr(df["Annual Revenue"]), 3))
print()
print(df.groupby("Lead Tier")["Annual Revenue"].mean().round(2))

Lead tier distribution:
Lead Tier
Warm    140
Hot     104
Cold      6
Name: count, dtype: int64

Score vs. Annual Revenue correlation: 0.691

Lead Tier
Cold    12027.52
Hot     54135.41
Warm    22260.75
Name: Annual Revenue, dtype: float64


## 6. Full Request Pipeline


In [10]:
def process_lead(raw_payload):
    missing_fields = [f for f in REQUIRED_FIELDS if raw_payload.get(f) is None or raw_payload.get(f) == ""]
    if missing_fields:
        return {"success": False, "error": f"Missing required fields: {', '.join(missing_fields)}"}

    data = clean_data(raw_payload)
    email = data["email"]

    if not is_valid_email_syntax(email):
        return {"success": False, "error": "Invalid email syntax format."}

    lead_score, lead_tier, qualification_status, is_personal_email = calculate_lead_score(data)
    predicted_revenue = predict_revenue(data, lead_score)

    domain = get_email_domain(email)
    company_domain_display = "Personal Email" if is_personal_email else domain

    enriched_payload = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "first_name": data.get("first_name"),
        "last_name": data.get("last_name"),
        "full_name": f"{data.get('first_name')} {data.get('last_name')}",
        "email": email,
        "company_name": data.get("company_name"),
        "company_domain": company_domain_display,
        "is_personal_email": is_personal_email,
        "current_billing_tool": data.get("current_billing_tool"),
        "monthly_billing_volume": data.get("monthly_billing_volume"),
        "job_title": data.get("job_title"),
        "job_title_other": data.get("job_title_other", ""),
        "crm_stack": data.get("crm_stack"),
        "crm_stack_other": data.get("crm_stack_other", ""),
        "company_size": data.get("company_size"),
        "industry": data.get("industry"),
        "industry_other": data.get("industry_other", ""),
        "primary_revenue_challenge": data.get("primary_revenue_challenge"),
        "primary_revenue_challenge_other": data.get("primary_revenue_challenge_other", ""),
        "heard_from": data.get("heard_from", ""),
        "lead_score": lead_score,
        "lead_tier": lead_tier,
        "qualification_status": qualification_status,
        "predicted_revenue": predicted_revenue
    }
    return {"success": True, "processed_lead": enriched_payload}

In [11]:
sample_submission = {
    "first_name": "praveen", "last_name": "sambasivam", "email": "praveen@acmesaas.com",
    "company_name": "acme saas", "current_billing_tool": "legacy-enterprise",
    "monthly_billing_volume": 180000, "job_title": "cfo_head_of_finance",
    "crm_stack": "salesforce", "company_size": "1000_plus", "industry": "ai-cloud",
    "primary_revenue_challenge": "revenue-recognition-compliance", "heard_from": "linkedin"
}
process_lead(sample_submission)

{'success': True,
 'processed_lead': {'timestamp': '2026-08-09T08:53:18.260305+00:00',
  'first_name': 'Praveen',
  'last_name': 'Sambasivam',
  'full_name': 'Praveen Sambasivam',
  'email': 'praveen@acmesaas.com',
  'company_name': 'Acme Saas',
  'company_domain': 'acmesaas.com',
  'is_personal_email': False,
  'current_billing_tool': 'legacy-enterprise',
  'monthly_billing_volume': 180000,
  'job_title': 'cfo_head_of_finance',
  'job_title_other': '',
  'crm_stack': 'salesforce',
  'crm_stack_other': '',
  'company_size': '1000_plus',
  'industry': 'ai-cloud',
  'industry_other': '',
  'primary_revenue_challenge': 'revenue-recognition-compliance',
  'primary_revenue_challenge_other': '',
  'heard_from': 'linkedin',
  'lead_score': 100,
  'lead_tier': 'Hot',
  'qualification_status': 'MQL',
  'predicted_revenue': 69101.45}}

## 7. Running This as a Live Server (from inside this notebook)

In [12]:
import threading
from flask import Flask, request, jsonify
from flask_cors import CORS

live_app = Flask(__name__)
CORS(live_app)   # required -- without this the browser blocks the webpage's fetch() call

@live_app.route("/health", methods=["GET"])
def health_check():
    return jsonify({
        "status": "online",
        "make_webhook_configured": "YOUR_CUSTOM_WEBHOOK_ID" not in MAKE_WEBHOOK_URL,
        "model_loaded": _model is not None,
    }), 200

@live_app.route("/api/v1/leads/score", methods=["POST"])
def score_lead_endpoint():
    raw_payload = request.get_json(silent=True)
    if not raw_payload:
        return jsonify({"success": False, "error": "Invalid or missing JSON payload."}), 400

    result = process_lead(raw_payload)
    if not result["success"]:
        return jsonify(result), 400

    # Forward to Make.com, same as app.py does
    enriched = result["processed_lead"]
    make_posted, make_error = False, None
    if MAKE_WEBHOOK_URL and "YOUR_CUSTOM_WEBHOOK_ID" not in MAKE_WEBHOOK_URL:
        try:
            res = requests.post(MAKE_WEBHOOK_URL, json=enriched, timeout=5)
            make_posted = res.status_code == 200
            if not make_posted:
                make_error = f"Make.com returned status {res.status_code}"
        except requests.RequestException as err:
            make_error = str(err)
    else:
        make_error = "Make.com webhook URL not configured."

    return jsonify({**result, "make_webhook_dispatched": make_posted, "make_error": make_error}), 200

In [13]:
def _run_server():
    live_app.run(host="0.0.0.0", port=5051, debug=False, use_reloader=False)

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

import time
time.sleep(1.5)  # give it a moment to bind the port

# Confirm it's actually up
try:
    r = requests.get("http://localhost:5051/health", timeout=3)
    print("Server is live:", r.json())
    print("\nOpen billswift-webinar-registration.html in a browser now -- submissions will hit this notebook.")
except Exception as e:
    print("Server did not start correctly:", e)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5051 is in use by another program. Either identify and stop that program, or start the server with a different port.


Server is live: {'make_webhook_configured': True, 'model_loaded': True, 'status': 'online'}

Open billswift-webinar-registration.html in a browser now -- submissions will hit this notebook.


Press CTRL+C to quit

127.0.0.1 - - [08/Aug/2026 04:33:43] "GET /health HTTP/1.1" 200 -


Server is live: {'make_webhook_configured': True, 'model_loaded': True, 'status': 'online'}

Open billswift-webinar-registration.html in a browser now -- submissions will hit this notebook.


In [14]:
# Prove the actual scoring endpoint works over real HTTP, not just as a
# direct function call -- this is exactly what the webinar page will do.
test_payload = {
    "first_name": "Sarah", "last_name": "Chen", "email": "sarah.chen@acmecloud.com",
    "company_name": "Acme Cloud Inc.", "industry": "ai-cloud",
    "job_title": "cfo_head_of_finance", "company_size": "1000_plus",
    "crm_stack": "salesforce", "monthly_billing_volume": "250000",
    "primary_revenue_challenge": "manual-no-automation",
    "current_billing_tool": "legacy-enterprise", "heard_from": "linkedin",
}
r = requests.post("http://localhost:5051/api/v1/leads/score", json=test_payload, timeout=5)
print("Status:", r.status_code)
r.json()

Status: 200


{'make_error': None,
 'make_webhook_dispatched': True,
 'processed_lead': {'company_domain': 'acmecloud.com',
  'company_name': 'Acme Cloud Inc.',
  'company_size': '1000_plus',
  'crm_stack': 'salesforce',
  'crm_stack_other': '',
  'current_billing_tool': 'legacy-enterprise',
  'email': 'sarah.chen@acmecloud.com',
  'first_name': 'Sarah',
  'full_name': 'Sarah Chen',
  'heard_from': 'linkedin',
  'industry': 'ai-cloud',
  'industry_other': '',
  'is_personal_email': False,
  'job_title': 'cfo_head_of_finance',
  'job_title_other': '',
  'last_name': 'Chen',
  'lead_score': 100,
  'lead_tier': 'Hot',
  'monthly_billing_volume': '250000',
  'predicted_revenue': 74095.43,
  'primary_revenue_challenge': 'manual-no-automation',
  'primary_revenue_challenge_other': '',
  'qualification_status': 'MQL',
  'timestamp': '2026-08-09T08:53:20.848551+00:00'},
 'success': True}